<a href="https://colab.research.google.com/github/asad1gove/bio-inspired-systems-lab/blob/main/cuckooSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np

cost_coefs = np.array([
    [0.00156, 7.92, 561],
    [0.00194, 7.85, 310],
    [0.00482, 7.97, 78]
])
p_min = np.array([150.0, 100.0, 50.0])
p_max = np.array([600.0, 400.0, 200.0])
PD = 850.0

def calculate_cost(P):
    return np.sum(cost_coefs[:, 0] * (P**2) + cost_coefs[:, 1] * P + cost_coefs[:, 2])

def fitness_function(P):
    P_clipped = np.clip(P, p_min, p_max)
    mismatch = np.sum(P_clipped) - PD
    return calculate_cost(P_clipped) + 10000.0 * (mismatch ** 2)

def run_simple_cuckoo(n_nests=20, max_iter=100, Pa=0.25):
    nests = np.random.uniform(p_min, p_max, (n_nests, 3))
    costs = np.array([fitness_function(n) for n in nests])

    best_idx = np.argmin(costs)
    best_nest = nests[best_idx].copy()
    best_cost = costs[best_idx]

    for iteration in range(max_iter):
        for i in range(n_nests):
            step = np.random.normal(0, 0.1, size=3) * (nests[i] - best_nest)
            new_nest = np.clip(nests[i] + step * np.random.normal(3), p_min, p_max)
            new_cost = fitness_function(new_nest)
            if new_cost < costs[i]:
                nests[i] = new_nest
                costs[i] = new_cost

        for i in range(n_nests):
            if np.random.rand() < Pa:
                nests[i] = np.random.uniform(p_min, p_max)
                costs[i] = fitness_function(nests[i])

        best_idx = np.argmin(costs)
        if costs[best_idx] < best_cost:
            best_cost = costs[best_idx]
            best_nest = nests[best_idx].copy()

    return np.clip(best_nest, p_min, p_max)

best_allocation = run_simple_cuckoo()

print("OUTPUTS:")
for idx, val in enumerate(best_allocation):
    print(f"Generator {idx+1}: {val:.2f} MW")
print(f"Total Generation: {np.sum(best_allocation):.2f} MW")
print(f"Total Cost: ${calculate_cost(best_allocation):.2f}/hr")

OUTPUTS:
Generator 1: 451.43 MW
Generator 2: 315.22 MW
Generator 3: 83.40 MW
Total Generation: 850.04 MW
Total Cost: $8207.61/hr
